In [0]:
from pyspark.sql.functions import current_date, lit

# 1. We take our cleaned Silver data
silver_df = spark.read.table("f1_processed.silver_cleaned")

# 2. We add our 'Historian' columns to it
# For the first load, everyone is 'Current' and has no 'End Date'
gold_driver_standings = silver_df.withColumn("start_date", current_date()) \
                                 .withColumn("end_date", lit(None).cast("date")) \
                                 .withColumn("is_current", lit(True))

# 3. Save it as our Gold Master Table
gold_driver_standings.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("f1_processed.gold_driver_history")

print("✅ Gold 'Historian' Table created with SCD Type 2 columns!")

In [0]:
%sql
SELECT name 
FROM f1_processed.gold_driver_history 
WHERE driverid = 44 
  AND '2020-06-15' BETWEEN start_date AND end_date

In [0]:
display(spark.table("f1_processed.gold_driver_history"))

In [0]:
# 1. Force read the Silver table (using batch mode to ensure we get everything)
silver_df = spark.read.table("f1_processed.silver_cleaned")

# 2. Check: Does Silver actually have data?
print(f"Total rows in Silver: {silver_df.count()}")

# 3. If count > 0, let's build Gold again
if silver_df.count() > 0:
    gold_driver_standings = silver_df.withColumn("start_date", current_date()) \
                                     .withColumn("end_date", lit(None).cast("date")) \
                                     .withColumn("is_current", lit(True))

    # We use 'overwrite' to ensure we start fresh
    gold_driver_standings.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("f1_processed.gold_driver_history")
    
    print("✅ Gold Table populated successfully!")
else:
    print("⚠️ Silver table is empty! We need to re-run your Silver ingestion cell first.")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date, lit

# 1. Load the "Target" (Your Gold Table)
gold_table = DeltaTable.forName(spark, "workspace.f1_processed.gold_driver_history")

# 2. Load the "Source" (Your New/Updated Silver Data)
updates_df = spark.read.table("f1_processed.silver_cleaned").dropDuplicates(["driverid"])

# 3. The MERGE Logic
gold_table.alias("target").merge(
    updates_df.alias("source"),
    "target.driverid = source.driverid"
).whenMatchedUpdate(
    # If the constructorId changed, expire the old row!
    condition = "target.constructorId <> source.constructorId AND target.is_current = true",
    set = {
        "is_current": "false",
        "end_date": current_date()
    }
).whenNotMatchedInsert(
    # If it's a new driver, add them as current!
    values = {
        "driverid": "source.driverid",
        "name": "source.name",
        "constructorId": "source.constructorId",
        "is_current": lit(True),
        "start_date": current_date(),
        "end_date": lit(None)
    }
).execute()

print("🏁 SCD Type 2 Merge Executed! Your history is now protected.")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date, lit, col

# 1. Target & Source
target_table_name = "f1_processed.gold_driver_history"
gold_table = DeltaTable.forName(spark, target_table_name)

# 2. Pre-process Source: Ensure only ONE row per driver from Silver
# We take the latest record if there are duplicates in Silver
source_df = spark.read.table("f1_processed.silver_cleaned") \
    .dropDuplicates(["driverid"]) 

# 3. The "Atomic" Merge
# Instead of trying to do everything in one complex join, 
# we use the standard 'Upsert' logic first.
gold_table.alias("target").merge(
    source_df.alias("source"),
    "target.driverid = source.driverid AND target.is_current = true"
).whenMatchedUpdate(
    # If the team (constructorId) changed, mark the old row as expired
    condition = "target.constructorId <> source.constructorId",
    set = {
        "is_current": "false",
        "end_date": current_date()
    }
).whenNotMatchedInsert(
    # If the driver isn't in Gold yet, or we're adding a new version
    values = {
        "driverid": "source.driverid",
        "name": "source.name",
        "constructorId": "source.constructorId",
        "is_current": lit(True),
        "start_date": current_date(),
        "end_date": lit(None)
    }
).execute()

print("🏁 SCD Type 2 Merge Executed successfully without conflicts!")

In [0]:
display(spark.table("f1_processed.gold_driver_history").orderBy("driverid"))

In [0]:
spark.sql("DROP TABLE IF EXISTS f1_processed.gold_driver_history")

In [0]:
from pyspark.sql.functions import current_date, lit, col

# 1. Load your clean Silver data
# We filter out any rows where driverid is already null to be safe
silver_df = spark.read.table("f1_processed.silver_cleaned").filter(col("driverid").isNotNull())

# 2. Add our 'Historian' columns
gold_df = silver_df.select(
    col("driverid"), 
    col("name"), 
    col("constructorId")
).withColumn("start_date", current_date()) \
 .withColumn("end_date", lit(None).cast("date")) \
 .withColumn("is_current", lit(True))

# 3. Save it fresh
gold_df.write.format("delta").mode("overwrite").saveAsTable("f1_processed.gold_driver_history")

print(f"✅ Reset Complete. Table recreated with {gold_df.count()} rows.")

In [0]:
# Check for any NULLs in the driverid column
null_count = spark.table("f1_processed.gold_driver_history").filter(col("driverid").isNull()).count()

if null_count == 0:
    print("🚀 Perfect! Every driver has an ID. No NULLs found.")
    display(spark.table("f1_processed.gold_driver_history").limit(10))
else:
    print(f"⚠️ Warning: Still found {null_count} rows with NULL driverid.")

In [0]:
OPTIMIZE f1_processed.gold_driver_history
ZORDER BY (driverid)

In [0]:
# Optimize the table and Z-Order by driverid for faster searches
spark.sql("OPTIMIZE f1_processed.gold_driver_history ZORDER BY (driverid)")

# Show the results of the optimization (How many files were removed/added)
display(spark.sql("DESCRIBE HISTORY f1_processed.gold_driver_history"))